# [12-1강] CNN 모델 설계 기준 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. 입력 channel에 맞는 첫 Conv layer 선택하기

이미지 데이터가 grayscale인지 RGB인지에 따라 첫 Conv2d의 `in_channels`가 달라집니다.

In [4]:
gray_batch = torch.randn(4, 1, 16, 16)
rgb_batch = torch.randn(4, 3, 16, 16)

# TODO: 각 입력 channel 수에 맞게 수정하세요.
gray_stem = nn.Conv2d(1, 8, kernel_size=3, padding=1)
rgb_stem = nn.Conv2d(3, 8, kernel_size=3, padding=1)

for name, stem, batch in [('gray', gray_stem, gray_batch), ('rgb', rgb_stem, rgb_batch)]:
    try:
        print(name, stem(batch).shape)
    except RuntimeError as e:
        print(name, '수정 필요:', str(e).split('\\n')[0])


gray torch.Size([4, 8, 16, 16])
rgb torch.Size([4, 8, 16, 16])


## 문제 2. filter progression이 있는 Conv block 만들기

CNN에서는 보통 뒤로 갈수록 channel 수를 늘려 더 많은 feature를 표현합니다.

In [7]:
# TODO: 1 -> 8 -> 16 channel 구조를 완성하세요.
features = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)
x = torch.randn(2, 1, 16, 16)
y = features(x)
print(y.shape)


torch.Size([2, 16, 4, 4])


### 해설 및 실행 결과 해석

- 두 번의 2x2 pooling으로 16x16 공간 크기가 8x8, 다시 4x4로 줄어듭니다. channel은 16개가 되어 flatten 전 feature는 `[N, 16, 4, 4]`가 됩니다.

## 문제 3. 설계안에서 classifier 입력 차원 자동 계산하기

이미지 크기가 바뀌어도 dummy 입력으로 classifier 입력 차원을 자동 계산할 수 있습니다.

In [13]:
features = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)
dummy = torch.randn(3, 1, 20, 20)

# TODO: dummy를 통과시켜 classifier 입력 차원을 계산하세요.
feat = features(dummy)

in_features = feat.view(feat.size(0), -1).shape[-1]
classifier = nn.Linear(in_features, 5)
try:
    feat = features(dummy)
    logits = classifier(feat.view(feat.size(0), -1))
    print(logits.shape)
except RuntimeError as e:
    print('classifier 입력 차원을 다시 계산하세요:', str(e).split('\\n')[0])


torch.Size([3, 5])


In [8]:
####

features = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)
dummy = torch.randn(3, 1, 20, 20)
feat = features(dummy)
in_features = feat.view(feat.size(0), -1).shape[1]
classifier = nn.Linear(in_features, 5)
logits = classifier(feat.view(feat.size(0), -1))
print('feature shape:', feat.shape)
print('in_features:', in_features)
print('logits shape:', logits.shape)


feature shape: torch.Size([3, 16, 5, 5])
in_features: 400
logits shape: torch.Size([3, 5])


### 해설 및 실행 결과 해석

- dummy 기반 계산은 CNN 설계에서 안전한 방법입니다. 입력 이미지 크기가 바뀌어도 flatten 크기를 코드가 직접 계산하므로 Linear dimension mismatch를 줄일 수 있습니다.